🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
🧩 Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:

Course policies (refunds, deadlines)
Lecture content (PDF notes)
Assignment deadlines
Their enrollment status
The company wants a single intelligent assistant that:

Answers questions using internal documents (PDFs, FAQs)
Fetches student-specific data (like enrollment or progress) using tools/APIs
Avoids hallucination and gives reliable answers
💻 Task
Design and implement a working prototype (pseudo-code or real code) for this system.

Your solution must include:

✅ 1. RAG Pipeline
How you will ingest and preprocess documents
Chunking strategy (with justification)
Embedding + vector store choice
Retrieval logic
How context is injected into the LLM
✅ 2. Agent Integration
Design an agent that decides:
When to use RAG
When to call a tool (e.g., get_student_status(student_id))
Show how tools are defined and used
✅ 3. End-to-End Flow
Write code or structured pseudo-code showing:

Input query
Retrieval step
Tool calling (if needed)
Final answer generation
✅ 4. Reliability Improvements
Show at least 2 techniques in code/design to:

Reduce hallucination
Improve answer quality
🎯 Expected Outcome
A working architecture/code that demonstrates:

RAG + Agent working together
Decision-making capability
Real-world practicality

In [12]:

from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter



# Groq Client (OpenAI-compatible)
client = OpenAI(
    api_key=("Api key"),
    base_url="https://api.groq.com/openai/v1"
)

def call_llm(system_prompt: str, user_message: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

print("Testing Groq...")
print(call_llm("You are a helpful assistant.", "Say hello in one sentence."))

DOCS = [
    {
        "text": (
            "Refund Policy: Students can get a full refund within 7 days of enrollment. "
            "Between 7-30 days, a 50% partial refund is available. "
            "No refunds after 30 days. Email support@edtech.com with your student ID."
        ),
        "source": "course_policy.pdf", "page": 1, "type": "policy"
    },
    {
        "text": (
            "Assignment Deadlines: Assignment 1 (Linear Regression) due March 31 2025. "
            "Assignment 2 (Neural Networks) due April 15 2025. "
            "Late penalty is 10% per day. Request extensions 48hrs in advance via the portal."
        ),
        "source": "course_policy.pdf", "page": 2, "type": "policy"
    },
    {
        "text": (
            "Lecture 3 - Neural Networks: A neural network has an input layer, hidden layers, "
            "and an output layer. Hidden layers use ReLU or Sigmoid activations. "
            "Training uses backpropagation and gradient descent to minimize loss."
        ),
        "source": "lecture3.pdf", "page": 1, "type": "lecture"
    },
    {
        "text": (
            "Overfitting: A model memorizes training data and fails on new data. "
            "Solutions: dropout, L2 regularization, early stopping, data augmentation."
        ),
        "source": "lecture3.pdf", "page": 2, "type": "lecture"
    },
    {
        "text": (
            "FAQ - Enrollment: Go to the course catalog, select a course, click Enroll. "
            "You need an active account and a valid payment method. "
            "Confirmation email arrives within 5 minutes."
        ),
        "source": "faq.pdf", "page": 1, "type": "faq"
    },
]

STUDENT_DB = {
    "S001": {"name": "Ayan",  "enrolled": True,  "course": "ML Fundamentals", "progress": 68},
    "S002": {"name": "Priya", "enrolled": True,  "course": "Deep Learning",   "progress": 42},
    "S003": {"name": "Rahul", "enrolled": False, "course": None,              "progress": 0},
}
ASSIGNMENT_DB = {
    "A1": {"title": "Linear Regression",    "due": "2025-03-31"},
    "A2": {"title": "Neural Network Design", "due": "2025-04-15"},
}

print(f"Loaded {len(DOCS)} docs | {len(STUDENT_DB)} students | {len(ASSIGNMENT_DB)} assignments")


import chromadb
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
chunks = []
for doc in DOCS:
    for i, t in enumerate(splitter.split_text(doc["text"])):
        chunks.append({"text": t, "source": doc["source"],
                       "page": doc["page"], "type": doc["type"]})
print(f"Created {len(chunks)} chunks")

# BM25 index only — no heavy embedding model needed
bm25_index = BM25Okapi([c["text"].lower().split() for c in chunks])
print(f"Indexed {len(chunks)} chunks (BM25 only, no download)")

def retrieve(query, k=4):
    scores = bm25_index.get_scores(query.lower().split())
    top    = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"text": chunks[i]["text"],
             "meta": {"source": chunks[i]["source"], "page": chunks[i]["page"]}}
            for i in top]

hits = retrieve("refund policy")
print(f"\nTest 'refund policy' -> {len(hits)} chunks")
print("Top:", hits[0]["text"][:80], "...")


def get_student(sid):
    return STUDENT_DB.get(sid.upper(), {"error": f"Student {sid} not found"})

def get_assignment(aid):
    return ASSIGNMENT_DB.get(aid.upper(), {"error": f"Assignment {aid} not found"})

def check_refund(sid, days):
    if sid.upper() not in STUDENT_DB:
        return {"error": "Student not found"}
    tier = "full 100%" if days <= 7 else "partial 50%" if days <= 30 else "none 0%"
    return {"student": sid, "days": days, "refund": tier}

print(get_student("S001"))
print(get_assignment("A1"))
print(check_refund("S001", 10))

def route(query):
    q        = query.lower()
    personal = any(k in q for k in ["my enrollment","my progress","student id",
                                     "am i enrolled","refund eligible"])
    docbased = any(k in q for k in ["policy","how do i","what is","explain",
                                     "lecture","faq","enroll","refund policy"])
    if personal and docbased: return "both"
    if personal:              return "tool"
    return "rag"

def run_tool(query):
    q   = query.lower()
    m   = re.search(r'\b[Ss]\d{3}\b', query)
    sid = m.group(0).upper() if m else None
    m   = re.search(r'\b[Aa]\d\b', query)
    aid = m.group(0).upper() if m else None

    if sid and any(k in q for k in ["enrollment","progress","status"]):
        return get_student(sid)
    if aid and any(k in q for k in ["deadline","due"]):
        return get_assignment(aid)
    if sid and "refund" in q:
        m    = re.search(r'(\d+)\s*day', q)
        days = int(m.group(1)) if m else 15
        return check_refund(sid, days)
    return None

RAG_SYSTEM = (
    "You are EduBot, an AI assistant for an online learning platform.\n"
    "Answer ONLY from the CONTEXT below. Cite [source, page] for every fact.\n"
    "If the answer is not in context say: I don't have that info — contact support@edtech.com\n"
    "CONTEXT:\n{context}"
)

def ask(query):
    r = route(query)
    tool_out, rag_out = None, None
    print(f"  route={r.upper()}", end="")

    if r in ("tool", "both"):
        tool_out = run_tool(query)
        print(f" | tool={tool_out}", end="")

    if r in ("rag", "both"):
        hits    = retrieve(query)
        ctx     = "\n---\n".join(
            f"[{h['meta']['source']} p{h['meta']['page']}]\n{h['text']}" for h in hits
        )
        rag_out = call_llm(RAG_SYSTEM.format(context=ctx), query)

    if r == "tool":
        if not tool_out or "error" in tool_out:
            ans = f"Sorry — {tool_out}"
        else:
            ans = call_llm(
                "You are a helpful student support assistant.",
                f'Student asked: "{query}"\nData: {json.dumps(tool_out)}\n'
                "Write a friendly 2-sentence response. Don't mention tools or APIs."
            )
    elif r == "rag":
        ans = rag_out
    else:
        ans = call_llm(
            "You are a helpful student support assistant.",
            f'Question: "{query}"\nTool data: {json.dumps(tool_out)}\n'
            f'Doc answer: {rag_out}\nCombine into one clear response.'
        )
    print()
    return ans

print("Agent ready.")

def safe_answer(query, answer):
    hits = retrieve(query)

    # Technique 1: nothing retrieved — high hallucination risk
    if not hits:
        return "I don't have verified info on this. Please contact support@edtech.com."

    # Technique 2: model itself said it doesn't know
    if "don't have that info" in answer.lower():
        return (
            f"Couldn't find a confident answer for: '{query}'.\n"
            "Please check the student portal or email support@edtech.com."
        )
    return answer

print("Reliability checks ready.")

queries = [
    "What is the refund policy?",                            # rag
    "Explain how neural networks work.",                     # rag
    "What is the deadline for assignment A1?",               # tool
    "What is the enrollment status for student S001?",       # tool
    "Is student S001 eligible for a refund after 10 days?",  # tool
    "How do I enroll in a course?",                          # rag
]

for q in queries:
    print(f"\nQ: {q}")
    ans   = ask(q)
    final = safe_answer(q, ans)
    print(f"A: {final}")
print("Done.")

Testing Groq...
Hello, it's nice to meet you and I'm here to help with any questions or tasks you may have.
Loaded 5 docs | 3 students | 2 assignments
Created 5 chunks
Indexed 5 chunks (BM25 only, no download)

Test 'refund policy' -> 4 chunks
Top: Refund Policy: Students can get a full refund within 7 days of enrollment. Betwe ...
{'name': 'Ayan', 'enrolled': True, 'course': 'ML Fundamentals', 'progress': 68}
{'title': 'Linear Regression', 'due': '2025-03-31'}
{'student': 'S001', 'days': 10, 'refund': 'partial 50%'}
Agent ready.
Reliability checks ready.

Q: What is the refund policy?
  route=RAG
A: The refund policy is as follows: 
- Students can get a full refund within 7 days of enrollment. 
- Between 7-30 days, a 50% partial refund is available. 
- No refunds are available after 30 days. 
To request a refund, students should email support@edtech.com with their student ID [course_policy.pdf, p1].

Q: Explain how neural networks work.
  route=RAG
A: A neural network has an input lay